## Calibration of document parsing pipeline for data extraction

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import os.path as osp
from pathlib import Path
import sys

root = Path.cwd().parent 
if str(root) not in sys.path:
    sys.path.append(str(root))

In [3]:
from src.config import CFG

In [4]:
import pandas as pd
import pdfplumber

In [5]:
CFG.PROJECT_ROOT, CFG.DATA_DIR, CFG.PROCESSED_DATA_DIR

(PosixPath('/Users/dric/projects/research/LLMs/chat_app'),
 '/Users/dric/projects/research/LLMs/chat_app/data',
 '/Users/dric/projects/research/LLMs/chat_app/data/processed')

In [6]:
raw_data_path = osp.join(CFG.DATA_DIR, "raw")
raw_data_path

'/Users/dric/projects/research/LLMs/chat_app/data/raw'

In [7]:
pdf_path = osp.join(raw_data_path, os.listdir(raw_data_path)[0])

pdf_path

'/Users/dric/projects/research/LLMs/chat_app/data/raw/EDAN_2025_RESULTAT_NATIONAL_DETAILS.pdf'

## Data Extraction Pipeline

In [8]:
def get_data_from_pdf(
        pdf_path:str
    ):
    all_tables      = []
    all_tables_df   = []

    pdf_cols = [
        'REGION',
        'CIRCONSCRIPTION_NUM',
        'CIRCONSCRIPTION_TITLE',
        'NB_BV',
        'REGISTERED',
        'VOTERS',
        'PART_RATE',
        'NULL_BALL',
        'EXPRESSED_VOTES',
        'NB_BLANK',
        'PCT_BLANK',
        'PARTY_NAME',
        'CANDIDATE_NAME',
        'SCORES',
        'PCT_SCORE',
        'IS_WINNER'
    ]
    
    settings        = {
        "vertical_strategy": "lines",
        "horizontal_strategy": "lines", 
        "snap_tolerance": 3,            
        "join_tolerance": 3,            
        "intersection_tolerance": 10, 
    }
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables(table_settings=settings)
            for table in tables:
                all_tables.append(table)
                df = pd.DataFrame(table[2:], columns=table[:2])
                all_tables_df.append(df)
    
    try:
        for df in all_tables_df:
            df.columns = pdf_cols
            
        final_df            = pd.concat(all_tables_df, axis=0, ignore_index=True)
        totals              = final_df.iloc[0] # extract totals line from final table
        final_df            = final_df.drop(index=0).reset_index(drop=True)

        # fix region names
        final_df["REGION"]  = final_df["REGION"].apply(lambda r: clean_region_name(r))
        final_df            = final_df.ffill()

        return totals, final_df, all_tables, all_tables_df
    
    except Exception as e:
        print(f" Error while merging data: {e}")
        return None, None, all_tables, all_tables_df
    
def clean_region_name(region:str):
    if region is not None and "\n" in region:
        return region.replace("\n", "")[::-1]
    else:
        return region
    
def summarize_turnout(totals:pd.Series):
    turnout_summary = "These are the overall numbers from the 2025 legislative elections: "

    for k in totals.keys()[1:]:
        if totals[k] not in [None, ''] and totals[k]:
            entry=f"{k}={totals[k]}, "
            turnout_summary+=entry

    turnout_summary = turnout_summary.strip(' ,')

    return turnout_summary

In [9]:
totals, df, tables, tables_df = get_data_from_pdf(pdf_path)

if df is not None:
    print(df.shape)

(1124, 16)


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1124 entries, 0 to 1123
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   REGION                 1124 non-null   object
 1   CIRCONSCRIPTION_NUM    1124 non-null   object
 2   CIRCONSCRIPTION_TITLE  1124 non-null   object
 3   NB_BV                  1124 non-null   object
 4   REGISTERED             1124 non-null   object
 5   VOTERS                 1124 non-null   object
 6   PART_RATE              1124 non-null   object
 7   NULL_BALL              1124 non-null   object
 8   EXPRESSED_VOTES        1124 non-null   object
 9   NB_BLANK               1124 non-null   object
 10  PCT_BLANK              1124 non-null   object
 11  PARTY_NAME             1124 non-null   object
 12  CANDIDATE_NAME         1124 non-null   object
 13  SCORES                 1124 non-null   object
 14  PCT_SCORE              1124 non-null   object
 15  IS_WINNER            

In [11]:
df.describe().T

,count,unique,top,freq
REGION,1124,34,,136
CIRCONSCRIPTION_NUM,1124,205,,20
CIRCONSCRIPTION_TITLE,1124,206,"BAYOTA, DAHIEPA·KEHI, OURAGAHIO ET YOPOHUE,\nC...",17
NB_BV,1124,112,68,33
REGISTERED,1124,203,42 324,17
VOTERS,1124,203,10693,17
PART_RATE,1124,202,"25,26%",17
NULL_BALL,1124,175,246,19
EXPRESSED_VOTES,1124,204,10 419,17
NB_BLANK,1124,129,70,31


In [12]:
def get_avg_vals(df:pd.DataFrame, col_name:str, dtype:str='int'):

    if dtype=='int':
        l = [int(n.replace(' ', '')) for n in df[col_name] if n!='']
        avg = sum(l)//len(l)
    else:
        l = [float(pct.replace('%', '').replace(',', '.')) for pct in df[col_name]]
        avg = sum(l)/len(l)
    
    return avg



In [13]:
get_avg_vals(df, 'NB_BV'), get_avg_vals(df, 'PART_RATE', dtype='float')

(127, 37.19553380782918)

In [14]:
num_cols = [
    'NB_BV',
    'REGISTERED',
    'VOTERS',
    'NULL_BALL',
    'EXPRESSED_VOTES',
    'NB_BLANK',
    'SCORES',
]

pct_cols = [
    'PART_RATE',
    'PCT_BLANK',
    'PCT_SCORE',
]

# process int-based columns
for col in num_cols:
    df[col] = df[col].apply(lambda n: n.replace(' ', ''))
    df[col] = df[col].replace('', get_avg_vals(df, col, 'int'))

# process float-based columns
for col in pct_cols:
    df[col] = df[col].apply(lambda n: n.replace('%', '').replace(',', '.'))
    df[col] = df[col].replace('', get_avg_vals(df, col, 'float'))

In [15]:
df.head(2)

,REGION,CIRCONSCRIPTION_NUM,CIRCONSCRIPTION_TITLE,NB_BV,REGISTERED,VOTERS,PART_RATE,NULL_BALL,EXPRESSED_VOTES,NB_BLANK,PCT_BLANK,PARTY_NAME,CANDIDATE_NAME,SCORES,PCT_SCORE,IS_WINNER
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,KOTO EHOU SOPIE,547,4.00,
1,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,RHDP,KOFFI AKA CHARLES,9078,66.35,ELU(E)


In [16]:
df = df.astype({
    'REGION':str,
    'CIRCONSCRIPTION_NUM':str,
    'CIRCONSCRIPTION_TITLE':str,
    'NB_BV':int,
    'REGISTERED':int,
    'VOTERS':int,
    'PART_RATE':float,
    'NULL_BALL':int,
    'EXPRESSED_VOTES':int,
    'NB_BLANK':int,
    'PCT_BLANK':float,
    'PARTY_NAME':str,
    'CANDIDATE_NAME':str,
    'SCORES':int,
    'PCT_SCORE':float,
    'IS_WINNER':bool
})

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1124 entries, 0 to 1123
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   REGION                 1124 non-null   object 
 1   CIRCONSCRIPTION_NUM    1124 non-null   object 
 2   CIRCONSCRIPTION_TITLE  1124 non-null   object 
 3   NB_BV                  1124 non-null   int64  
 4   REGISTERED             1124 non-null   int64  
 5   VOTERS                 1124 non-null   int64  
 6   PART_RATE              1124 non-null   float64
 7   NULL_BALL              1124 non-null   int64  
 8   EXPRESSED_VOTES        1124 non-null   int64  
 9   NB_BLANK               1124 non-null   int64  
 10  PCT_BLANK              1124 non-null   float64
 11  PARTY_NAME             1124 non-null   object 
 12  CANDIDATE_NAME         1124 non-null   object 
 13  SCORES                 1124 non-null   int64  
 14  PCT_SCORE              1124 non-null   float64
 15  IS_W

In [18]:
df.head(10)

,REGION,CIRCONSCRIPTION_NUM,CIRCONSCRIPTION_TITLE,NB_BV,REGISTERED,VOTERS,PART_RATE,NULL_BALL,EXPRESSED_VOTES,NB_BLANK,PCT_BLANK,PARTY_NAME,CANDIDATE_NAME,SCORES,PCT_SCORE,IS_WINNER
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,KOTO EHOU SOPIE,547,4.00,False
1,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,RHDP,KOFFI AKA CHARLES,9078,66.35,True
2,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,YEPIE KOUASSI THEODORE,58,0.42,False
3,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,TCHIMOU GNAMON BERTRAND,1991,14.55,False
4,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,BOKA BOKA MAURICE,674,4.93,False
5,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,ADCI,EDI DOFFOU PAUL,331,2.42,False
6,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,FPI,N'GUESSAN KOTCHI REMI,474,3.46,False
7,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,MGC,NANDO YAVO SERGE,453,3.31,False
8,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE,133,48710,12821,26.32,317,12504,81,0.65,ADCI,OCHOU WROHOUM MARIE-PASCALE,296,2.37,False
9,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE,133,48710,12821,26.32,317,12504,81,0.65,RHDP,DIMBA N'GOU PIERRE,10675,85.37,True


### Save procesed dataframe

In [19]:
df.to_csv(osp.join(CFG.PROCESSED_DATA_DIR, '2025_legislative_elections.csv'), index=False)

### Turnout summary from totals

In [20]:
summarize_turnout(totals)

'These are the overall numbers from the 2025 legislative elections: NB_BV=25 338, REGISTERED=8 597 092, VOTERS=3 012 094, PART_RATE=35,04%, NULL_BALL=68 525, EXPRESSED_VOTES=2 943 569, NB_BLANK=29 578, PCT_BLANK=1,00%, SCORES=2 913 991'

In [21]:
# Group by everything that identifies the location
group_cols = ['REGION', 'CIRCONSCRIPTION_NUM', 'CIRCONSCRIPTION_TITLE', 'NB_BV', 'REGISTERED', 'VOTERS',
              'PART_RATE', 'NULL_BALL', 'EXPRESSED_VOTES', 'NB_BLANK', 'PCT_BLANK']

df_grouped = df.groupby(group_cols).agg({
    'PARTY_NAME': list,
    'CANDIDATE_NAME': list,
    'SCORES': list,
    'PCT_SCORE': list,
    'IS_WINNER': 'first'
}).reset_index()


In [22]:
df_grouped

,REGION,CIRCONSCRIPTION_NUM,CIRCONSCRIPTION_TITLE,NB_BV,REGISTERED,VOTERS,PART_RATE,NULL_BALL,EXPRESSED_VOTES,NB_BLANK,PCT_BLANK,PARTY_NAME,CANDIDATE_NAME,SCORES,PCT_SCORE,IS_WINNER
0,,,,127,31758,9221,29.04,283,8938,72,0.81,[RHDP],[KANGBE YAYORO CHARLES LOPEZ],[3169],[35.46],True
1,,,,127,39362,16841,42.78,518,16323,118,0.72,[PDCI-RDA],[MIAN OCTAVI PEROU],[159],[0.97],False
2,,,PREFECTURES,127,26419,11837,44.80,202,11635,228,1.96,"[INDEPENDANT, INDEPENDANT, CODE]","[KONE BAKARY, MOUHON BOZON ELVIS DESTY, YOMI K...","[1490, 214, 64]","[12.81, 1.84, 0.55]",False
3,,005,"GOMON ET SIKENSI, COMMUNES ET SOUS-PREFECTURES",102,37720,10768,28.55,347,10421,134,1.29,"[INDEPENDANT, INDEPENDANT, INDEPENDANT, INDEPE...","[BOTTI ADJIRI BERNARD, YAO YAO ERIC ARMAND, BI...","[1127, 1710, 1417, 580, 3574, 157, 64, 32, 1626]","[10.81, 16.41, 13.6, 5.57, 34.3, 1.51, 0.61, 0...",False
4,,006,"GBOLOUVILLE ET N'DOUCI, COMMUNES ET SOUS-\nPRE...",94,31758,9221,29.04,283,8938,72,0.81,"[ADCI, INDEPENDANT, INDEPENDANT, INDEPENDANT, ...","[CAMARA ISSOUF, TANOH AMOI CHRISTIAN, GOUHAN E...","[1848, 88, 282, 868, 126, 110, 2375]","[20.68, 0.98, 3.16, 9.71, 1.41, 1.23, 26.57]",False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,WORODOUGOU,201,"KANI, COMMUNE ET SOUS-PREFECTURE",41,13143,7621,57.99,143,7478,115,1.54,"[INDEPENDANT, INDEPENDANT, RHDP, INDEPENDANT]","[YACOUBA MEITE, MEITE YAYA, MEITE BEN ABDOULAY...","[202, 4329, 2530, 302]","[2.7, 57.89, 33.83, 4.04]",False
213,WORODOUGOU,202,"BOBI-DIARABANA, COMMUNE ET SOUS-PREFECTURE,\nS...",60,16811,9578,56.97,69,9509,114,1.20,"[RHDP, INDEPENDANT, INDEPENDANT]","[DIOMANDE MAMADOU, FOFANA VATIECOUMBA, FOFANA ...","[6519, 2567, 309]","[68.56, 27.0, 3.25]",True
214,WORODOUGOU,203,"SEGUELA, COMMUNE",55,20148,7201,35.74,80,7121,171,2.40,"[INDEPENDANT, INDEPENDANT, RHDP, INDEPENDANT, ...","[SOUMAHORO ADAMA, MOUSTAPHA FALL, TIMITE SAHIN...","[357, 1214, 3038, 123, 2023, 195]","[5.01, 17.05, 42.66, 1.73, 28.41, 2.74]",False
215,WORODOUGOU,204,"DUALLA ET MASSALA, COMMUNES ET SOUS-\nPREFECTURES",48,12876,7334,56.96,85,7249,56,0.77,"[INDEPENDANT, INDEPENDANT, RHDP, INDEPENDANT, ...","[DOSSO ABOUBACAR SIDIKY, COULIBALY METOBA, DOS...","[2392, 2076, 2545, 7, 173]","[33.0, 28.64, 35.11, 0.1, 2.39]",False


In [23]:
df_grouped.shape

(217, 16)

### Turnout data

In [24]:
turnout_data = df[group_cols].drop_duplicates()

turnout_data

,REGION,CIRCONSCRIPTION_NUM,CIRCONSCRIPTION_TITLE,NB_BV,REGISTERED,VOTERS,PART_RATE,NULL_BALL,EXPRESSED_VOTES,NB_BLANK,PCT_BLANK
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56
8,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE,133,48710,12821,26.32,317,12504,81,0.65
14,AGNEBY-TIASSA,003,AZAGUIE COMMUNE ET SOUS-PREFECTURE,44,15515,5174,33.35,73,5101,24,0.47
22,AGNEBY-TIASSA,004,"ANANGUIE, CECHI ET RUBINO, COMMUNES ET SOUS-\n...",72,23466,7650,32.60,241,7409,49,0.66
32,,005,"GOMON ET SIKENSI, COMMUNES ET SOUS-PREFECTURES",102,37720,10768,28.55,347,10421,134,1.29
...,...,...,...,...,...,...,...,...,...,...,...
1101,WORODOUGOU,201,"KANI, COMMUNE ET SOUS-PREFECTURE",41,13143,7621,57.99,143,7478,115,1.54
1105,WORODOUGOU,202,"BOBI-DIARABANA, COMMUNE ET SOUS-PREFECTURE,\nS...",60,16811,9578,56.97,69,9509,114,1.20
1108,WORODOUGOU,203,"SEGUELA, COMMUNE",55,20148,7201,35.74,80,7121,171,2.40
1114,WORODOUGOU,204,"DUALLA ET MASSALA, COMMUNES ET SOUS-\nPREFECTURES",48,12876,7334,56.96,85,7249,56,0.77


In [25]:
turnout_data.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "turnout.parquet"), index=False)

### Results data

In [26]:
results_data = df[
    ['REGION', 'CIRCONSCRIPTION_TITLE', 'CIRCONSCRIPTION_NUM', 'PARTY_NAME', 'CANDIDATE_NAME', 'SCORES', 'PCT_SCORE', 'IS_WINNER']
]

results_data

,REGION,CIRCONSCRIPTION_TITLE,CIRCONSCRIPTION_NUM,PARTY_NAME,CANDIDATE_NAME,SCORES,PCT_SCORE,IS_WINNER
0,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,KOTO EHOU SOPIE,547,4.00,False
1,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,RHDP,KOFFI AKA CHARLES,9078,66.35,True
2,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,YEPIE KOUASSI THEODORE,58,0.42,False
3,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,TCHIMOU GNAMON BERTRAND,1991,14.55,False
4,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,BOKA BOKA MAURICE,674,4.93,False
...,...,...,...,...,...,...,...,...
1119,WORODOUGOU,"KAMALO, SIFIE ET WOROFLA, COMMUNES ET SOUS-\nP...",205,RHDP,MEITE ABOULAYE,11233,88.64,True
1120,WORODOUGOU,"KAMALO, SIFIE ET WOROFLA, COMMUNES ET SOUS-\nP...",205,INDEPENDANT,KONE LOSSENI,46,0.36,False
1121,WORODOUGOU,"KAMALO, SIFIE ET WOROFLA, COMMUNES ET SOUS-\nP...",205,INDEPENDANT,TRAORE SEKOU,646,5.10,False
1122,WORODOUGOU,"KAMALO, SIFIE ET WOROFLA, COMMUNES ET SOUS-\nP...",205,RDP,FOFANA SIAKA,175,1.38,False


In [27]:
results_data["IS_WINNER"] = results_data["IS_WINNER"].apply(lambda r: int(r=="ELU(E)"))

results_data.head()

/var/folders/w9/86zjmss95nv7d9kbd9x9lb_m0000gn/T/ipykernel_6642/1879952573.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_data["IS_WINNER"] = results_data["IS_WINNER"].apply(lambda r: int(r=="ELU(E)"))


,REGION,CIRCONSCRIPTION_TITLE,CIRCONSCRIPTION_NUM,PARTY_NAME,CANDIDATE_NAME,SCORES,PCT_SCORE,IS_WINNER
0,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,KOTO EHOU SOPIE,547,4.00,0
1,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,RHDP,KOFFI AKA CHARLES,9078,66.35,0
2,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,YEPIE KOUASSI THEODORE,58,0.42,0
3,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,TCHIMOU GNAMON BERTRAND,1991,14.55,0
4,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,BOKA BOKA MAURICE,674,4.93,0


In [28]:
results_data.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "results.parquet"), index=False)

### Region data

In [29]:
regions = pd.DataFrame(df["REGION"].unique(), columns=["REGION_NAME"])

regions

,REGION_NAME
0,AGNEBY-TIASSA
1,
2,BAFING
3,BAGOUE
4,BELIER
5,BERE
6,BOUNKANI
7,CAVALLY
8,DISTRICTAUTONOMED'ABIDJAN
9,DISTRICTAUTONOMEDEYAMOUSSOUKRO


In [30]:
regions.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "regions.parquet"), index=False)

### Circonsciption

In [31]:
circs = df[
    ["REGION", "CIRCONSCRIPTION_NUM", "CIRCONSCRIPTION_TITLE"]
].drop_duplicates().reset_index(drop=True)

circs

,REGION,CIRCONSCRIPTION_NUM,CIRCONSCRIPTION_TITLE
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO..."
1,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE
2,AGNEBY-TIASSA,003,AZAGUIE COMMUNE ET SOUS-PREFECTURE
3,AGNEBY-TIASSA,004,"ANANGUIE, CECHI ET RUBINO, COMMUNES ET SOUS-\n..."
4,,005,"GOMON ET SIKENSI, COMMUNES ET SOUS-PREFECTURES"
...,...,...,...
207,WORODOUGOU,201,"KANI, COMMUNE ET SOUS-PREFECTURE"
208,WORODOUGOU,202,"BOBI-DIARABANA, COMMUNE ET SOUS-PREFECTURE,\nS..."
209,WORODOUGOU,203,"SEGUELA, COMMUNE"
210,WORODOUGOU,204,"DUALLA ET MASSALA, COMMUNES ET SOUS-\nPREFECTURES"


In [32]:
circs["CIRCONSCRIPTION_TITLE"] = circs["CIRCONSCRIPTION_TITLE"].apply(lambda c: c.replace('-\n', '-').replace('\n', '').strip())
circs["LEN_CIRC"] = circs["CIRCONSCRIPTION_TITLE"].apply(lambda c: len(c))

circs

,REGION,CIRCONSCRIPTION_NUM,CIRCONSCRIPTION_TITLE,LEN_CIRC
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,LOVI...",121
1,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE,17
2,AGNEBY-TIASSA,003,AZAGUIE COMMUNE ET SOUS-PREFECTURE,34
3,AGNEBY-TIASSA,004,"ANANGUIE, CECHI ET RUBINO, COMMUNES ET SOUS-PR...",55
4,,005,"GOMON ET SIKENSI, COMMUNES ET SOUS-PREFECTURES",46
...,...,...,...,...
207,WORODOUGOU,201,"KANI, COMMUNE ET SOUS-PREFECTURE",32
208,WORODOUGOU,202,"BOBI-DIARABANA, COMMUNE ET SOUS-PREFECTURE,SEG...",67
209,WORODOUGOU,203,"SEGUELA, COMMUNE",16
210,WORODOUGOU,204,"DUALLA ET MASSALA, COMMUNES ET SOUS-PREFECTURES",47


In [33]:
circs.describe().T

,count,mean,std,min,25%,50%,75%,max
LEN_CIRC,212.0,50.872642,23.645442,0.0,38.5,52.0,65.0,130.0


In [34]:
circs[circs["LEN_CIRC"]>0].to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "circonscriptions.parquet"), index=False)

### Party

In [35]:
party_data = pd.DataFrame(df["PARTY_NAME"].unique(), columns=["PARTY_NAME"])

party_data.head()

,PARTY_NAME
0,INDEPENDANT
1,RHDP
2,ADCI
3,FPI
4,MGC


In [36]:
from unidecode import unidecode

party_data["PARTY_NORMALIZED"] = party_data["PARTY_NAME"].apply(lambda p: unidecode(p.lower().strip()))

In [37]:
party_data.head()

,PARTY_NAME,PARTY_NORMALIZED
0,INDEPENDANT,independant
1,RHDP,rhdp
2,ADCI,adci
3,FPI,fpi
4,MGC,mgc


In [38]:
party_data.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "parties.parquet"), index=False)

### Candidates data

In [39]:
candidates = df[
    ["PARTY_NAME","CANDIDATE_NAME", "CIRCONSCRIPTION_NUM"]
].drop_duplicates().reset_index(drop=True)

candidates

,PARTY_NAME,CANDIDATE_NAME,CIRCONSCRIPTION_NUM
0,INDEPENDANT,KOTO EHOU SOPIE,001
1,RHDP,KOFFI AKA CHARLES,001
2,INDEPENDANT,YEPIE KOUASSI THEODORE,001
3,INDEPENDANT,TCHIMOU GNAMON BERTRAND,001
4,INDEPENDANT,BOKA BOKA MAURICE,001
...,...,...,...
1119,RHDP,MEITE ABOULAYE,205
1120,INDEPENDANT,KONE LOSSENI,205
1121,INDEPENDANT,TRAORE SEKOU,205
1122,RDP,FOFANA SIAKA,205


In [40]:
candidates.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "candidates.parquet"), index=False)